# Phase 6: failure maps on half A

**CRISP-DM stage:** Evaluation

Heat maps were not saved in Phase 5, so each configuration (C1, C2, C4, C5) is fitted again with seed 42 and scores **half A only**. For each configuration and product, three kinds of photo are shown:

- **Hits:** the 2 defective photos with the highest scores, where the method is most sure.
- **Hardest defective photos:** the 4 with the lowest scores, the ones most likely to be missed.
- **Hardest good photos:** the 4 with the highest scores, the ones most likely to be false alarms.

Choosing by score, not by where a line sits, keeps these meaningful whichever line is used.

**A heat map shows where the model looked, not whether it was right to.**

**Runs on:** Kaggle **without a GPU**, with internet on. A GPU run of this notebook waited in Kaggle's queue for over an hour and a half, and this run only looks at failures, so it uses Kaggle's processor instead. Its numbers go into no results table, threshold or Final test. Processors and GPUs round numbers very slightly differently, so a score may differ in the third or fourth decimal place. On the laptop it only checks the code, with an untrained network on a few sample photos. Figures named `gallery_*` show real parts, so they stay out of git.

In [ ]:
!pip install anomalib==2.7.0

In [ ]:
import csv
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy
import torch
import yaml
from anomalib.data import MVTecAD2
from anomalib.models import Padim
from anomalib.models.image.padim.torch_model import PadimModel
from anomalib.models.image.patchcore.torch_model import PatchcoreModel
from torch.utils.data import DataLoader, Subset

# On Kaggle the data is in /kaggle/input, with or without a GPU
ON_KAGGLE = Path("/kaggle/input").exists()
USING_GPU = torch.cuda.is_available()
if USING_GPU:
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
if ON_KAGGLE:
    INPUT_FOLDER = Path("/kaggle/input")
    PRETRAINED = True
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 54, "vial": 70}
else:
    # The laptop only proves the code runs: tiny photos and an untrained network
    INPUT_FOLDER = Path("data/sample")
    PRETRAINED = False
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 2, "vial": 4}

# Finds the project's settings files: the private dataset on Kaggle, or the project folder on the laptop
code_folder = None
if ON_KAGGLE:
    for evaluate_file in Path("/kaggle/input").rglob("evaluate.py"):
        code_folder = evaluate_file.parent.parent
else:
    code_folder = Path(".")
if code_folder is None:
    raise FileNotFoundError("Could not find the project code")

CONFIGS = ["C1", "C2", "C4", "C5"]
# Seed 42 only: this run is for looking at failures, not for measuring results again
SEED = 42
HITS = 2
HARDEST = 4
# Half A, from half_a_files.csv. Every lighting version of each part is used
HALF_A_PARTS = {
    "sheet_metal": {"good": ["001", "002"], "bad": ["000", "004", "005", "007", "009", "012", "013"]},
    "vial": {"good": ["000", "002", "003"], "bad": ["001", "008", "010", "011", "012", "013", "014"]},
}
FIGURE_DPI = 120
RESULTS_FOLDER = Path("results")
RESULTS_FOLDER.mkdir(exist_ok=True)


def load_settings(config_id):
    """Read one configuration's settings file."""
    config_files = list((code_folder / "configs").glob(config_id + "_*.yaml"))
    with open(config_files[0], encoding="utf-8") as file:
        return yaml.safe_load(file)


def set_seed(seed):
    """Fix the random numbers used by Python, NumPy and PyTorch."""
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)


def build_model(method):
    """Build the model named in the settings, as in the Phase 5 notebooks."""
    if method["name"] == "padim":
        model = PadimModel(backbone=method["backbone"], layers=method["layers"], pre_trained=PRETRAINED,
                           n_features=method["n_features"])
    else:
        model = PatchcoreModel(backbone=method["backbone"], layers=method["layers"], pre_trained=PRETRAINED,
                               num_neighbors=method["num_neighbors"])
    return model.to(DEVICE)


def make_loader(dataset, batch_size, positions=None):
    """Build batches in a fixed order, optionally from chosen photos only."""
    source = dataset
    if positions is not None:
        source = Subset(dataset, positions)
    return DataLoader(source, batch_size=batch_size, collate_fn=dataset.collate_fn, shuffle=False)


def fit(model, method, train_data, recipe, batch_size):
    """Learn from good training photos, the same route as in Phase 5."""
    model.train()
    model.feature_extractor.eval()
    with torch.no_grad():
        for batch in make_loader(train_data, batch_size):
            model(recipe(batch.image).to(DEVICE))
    if method["name"] == "padim":
        embeddings = torch.vstack(model.memory_bank).cpu()
        model.memory_bank = []
        # Statistics on the processor, because the GPU's many-at-once inverse crashes in this setup
        model.gaussian.fit(embeddings)
        model.gaussian.to(DEVICE)
    else:
        model.subsample_embedding(method["coreset_sampling_ratio"])
    model.eval()


def score_photos(model, loader, recipe):
    """Return heat maps, photo scores, masks and paths."""
    maps = []
    scores = []
    masks = []
    paths = []
    with torch.no_grad():
        for batch in loader:
            images, batch_masks = recipe(batch.image, batch.gt_mask)
            output = model(images.to(DEVICE))
            maps.append(output.anomaly_map.squeeze(1).cpu())
            # A simple list of scores, to line up with the labels
            scores.append(output.pred_score.reshape(-1).cpu())
            masks.append(batch_masks)
            paths.extend(batch.image_path)
    return torch.cat(maps), torch.cat(scores), torch.cat(masks), paths

In [ ]:
picked_rows = []

for config_id in CONFIGS:
    settings = load_settings(config_id)
    method = settings["method"]
    image_size = settings["image"]["size"] if ON_KAGGLE else 64
    batch_size = method["train_batch_size"]
    recipe = Padim.configure_pre_processor(image_size=(image_size, image_size)).transform
    normalize_step = recipe.transforms[1]
    means = torch.tensor(normalize_step.mean).view(3, 1, 1)
    stds = torch.tensor(normalize_step.std).view(3, 1, 1)

    for product_name in ["sheet_metal", "vial"]:
        product_folder = None
        for train_folder in INPUT_FOLDER.rglob("train"):
            if train_folder.parent.name == product_name:
                product_folder = train_folder.parent
        datamodule = MVTecAD2(root=product_folder.parent, category=product_name, num_workers=0)
        datamodule.setup()

        half_a_positions = []
        test_paths = datamodule.test_public_data.samples["image_path"].tolist()
        for position in range(len(test_paths)):
            path = Path(test_paths[position])
            if path.stem.split("_")[0] in HALF_A_PARTS[product_name][path.parent.name]:
                half_a_positions.append(position)
        if len(half_a_positions) != EXPECTED_HALF_A_PHOTOS[product_name]:
            raise ValueError("Wrong number of half A photos for " + product_name)

        set_seed(SEED)
        model = build_model(method)
        fit(model, method, datamodule.train_data, recipe, batch_size)
        maps, scores, masks, paths = score_photos(
            model, make_loader(datamodule.test_public_data, batch_size, half_a_positions), recipe)

        defective = []
        good = []
        for position in range(len(paths)):
            if Path(paths[position]).parent.name == "bad":
                defective.append(position)
            else:
                good.append(position)

        def score_of(position):
            return float(scores[position])

        defective.sort(key=score_of)
        good.sort(key=score_of)
        chosen = []
        for position in defective[-HITS:]:
            chosen.append(("hit", position))
        for position in defective[:HARDEST]:
            chosen.append(("hardest defective", position))
        for position in good[-HARDEST:]:
            chosen.append(("hardest good", position))

        # One shared colour scale, so the maps in this figure can be compared
        top = float(maps.max())
        bottom = float(maps.min())
        figure, axes = plt.subplots(len(chosen), 2, figsize=(8, 3.6 * len(chosen)), squeeze=False)
        for row_number in range(len(chosen)):
            kind, position = chosen[row_number]
            path = Path(paths[position])
            photo = recipe(datamodule.test_public_data[half_a_positions[position]].image)
            photo = (photo * stds + means).clamp(0, 1)
            mask = masks[position].float()
            axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
            axes[row_number][1].imshow(maps[position], cmap="Blues", vmin=bottom, vmax=top)
            if bool(mask.any()):
                axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
                axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
            axes[row_number][0].set_title(kind + ": " + path.parent.name + " " + path.name, fontsize=9)
            axes[row_number][1].set_title("score " + str(round(float(scores[position]), 2)), fontsize=9)
            axes[row_number][0].axis("off")
            axes[row_number][1].axis("off")
            picked_rows.append([config_id, product_name, kind, path.parent.name, path.name, round(float(scores[position]), 4)])
        figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A, " + config_id
                        + ", seed 42. Red: mask. Darker blue: more suspicious", y=1.0)
        figure.tight_layout()
        figure.savefig("gallery_failures_" + config_id + "_" + product_name + ".png", dpi=FIGURE_DPI, bbox_inches="tight")
        # Closing without showing keeps dataset images out of the notebook's log
        plt.close(figure)
        print("Saved gallery for", config_id, product_name, flush=True)
        # Saved after every configuration, so a stop part-way loses nothing finished
        with open(RESULTS_FOLDER / "failure_picks.csv", "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["config", "product", "kind", "label", "file_name", "score"])
            for row in picked_rows:
                writer.writerow(row)
        del model
        if USING_GPU:
            torch.cuda.empty_cache()

# Numbers only, so this file can be committed
with open(RESULTS_FOLDER / "failure_picks.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["config", "product", "kind", "label", "file_name", "score"])
    for row in picked_rows:
        writer.writerow(row)
print("Finished")